In [ ]:
!pip install -q transformers peft accelerate datasets tqdm scikit-learn sentencepiece bitsandbytes

In [ ]:
import os
import gc
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    DataCollatorWithPadding, 
    get_linear_schedule_with_warmup,
    BitsAndBytesConfig  # <--- Added back
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import f1_score

# Clean memory start
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Get the token securely from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_KEY") 

# 2. Log in automatically (no popup box!)
login(token=hf_token)

In [ ]:
# 1. Load Data
dataset = load_dataset("ailsntua/QEvasion")

# 2. Labels
labels = sorted(dataset["train"].unique("clarity_label"))
num_labels = len(labels)
label2id = {l: i for i, l in enumerate(labels)}
dataset = dataset.map(lambda x: {"labels": label2id[x["clarity_label"]]})
dataset = dataset.remove_columns(["clarity_label"])

# 3. Text Merge
dataset = dataset.map(lambda x: {
    "text": f"Question: {x['interview_question']}\nAnswer: {x['interview_answer']}"
})

# 4. Tokenizer
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token 

# 5. Tokenize (Long Context)
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=True,
        max_length=1024  # <--- 1024 fits easily with 4-bit
    )

encoded = dataset.map(tokenize_fn, batched=True)
keep_cols = ["input_ids", "attention_mask", "labels"]
encoded = encoded.remove_columns([c for c in encoded["train"].column_names if c not in keep_cols])
encoded.set_format("torch")

# 6. DataLoaders
# 4-bit model is tiny (~2GB), so we can increase batch size
batch_size = 4
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(encoded["train"], batch_size=batch_size, shuffle=True, collate_fn=data_collator)
valid_loader = DataLoader(encoded["test"], batch_size=batch_size, collate_fn=data_collator)

print(f"Train batches: {len(train_loader)} (Batch size {batch_size}, Length 1024)")

In [ ]:
SAVE_PATH = "/kaggle/working/llama_focal_lora"
os.makedirs(SAVE_PATH, exist_ok=True)

# 1. Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load Model
print("Loading model in 4-bit...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    quantization_config=bnb_config, # <--- Apply Quantization
    num_labels=num_labels,
    device_map="auto"  # <--- Let Accelerate handle placement (likely GPU 0)
)
model.config.pad_token_id = tokenizer.pad_token_id

# 3. Prepare for Training (Casts layers to fp32 for stability)
model = prepare_model_for_kbit_training(model)

# 4. Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# NOTE: We do NOT use DataParallel here. 4-bit models don't support it easily.
# The model is likely on "cuda:0".

In [ ]:
gradient_accumulation_steps = 4 # Effective Batch = 16 (4 * 4)
lr = 2e-4 # QLoRA often prefers slightly higher LR
num_epochs = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=num_training_steps
)

best_f1 = 0
device = "cuda:0" # Main device

for epoch in range(1, num_epochs+1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    optimizer.zero_grad()
    
    for step, batch in enumerate(pbar):
        # Move to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        loss = outputs.loss / gradient_accumulation_steps
        loss.backward()
        
        if (step + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * gradient_accumulation_steps
        pbar.set_postfix({'loss': loss.item() * gradient_accumulation_steps})

    print(f"Epoch {epoch} | Loss: {total_loss / len(train_loader):.4f}")

    # Validation
    model.eval()
    preds, gts = [], []
    
    # Optional: Clear cache to be safe
    torch.cuda.empty_cache()

    with torch.no_grad():
        for batch in valid_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds.extend(torch.argmax(outputs.logits, dim=1).cpu().tolist())
            gts.extend(batch['labels'].cpu().tolist())

    val_f1 = f1_score(gts, preds, average='weighted')
    print(f"Validation F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        print("New best model! Saving...")
        model.save_pretrained(os.path.join(SAVE_PATH, "best_model"))
        tokenizer.save_pretrained(os.path.join(SAVE_PATH, "best_model"))